In [ ]:
from delta import *
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("GoldLayer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
)

from delta import configure_spark_with_delta_pip

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

In [ ]:
BASE = "/home/jovyan/work/delta_lake/silver"

orders       = spark.read.format("delta").load(f"{BASE}/orders")
order_items  = spark.read.format("delta").load(f"{BASE}/order_items")
products     = spark.read.format("delta").load(f"{BASE}/products")
customers    = spark.read.format("delta").load(f"{BASE}/customers")

# Đọc thẳng từ CSV vì chưa có trong silver
translation = spark.read.csv(
    "/home/jovyan/work/data/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

print("Load OK")
translation.show(5)

In [ ]:
from pyspark.sql import functions as F

gold_enriched = (
    order_items
    .join(orders,    "order_id")
    .join(products,  "product_id")
    .join(translation, "product_category_name", "left")
    .join(customers, "customer_id", "left")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "product_category_name",
        F.coalesce(
            translation["product_category_name_english"],  # chỉ rõ lấy từ bảng nào
            F.col("product_category_name")
        ).alias("category_english"),
        "price",
        "freight_value",
        "order_status",
        "order_purchase_timestamp",
        "customer_state",
    )
    .filter(F.col("order_status") == "delivered")
    .filter(F.col("category_english").isNotNull())
)

gold_enriched.write.format("delta").mode("overwrite") \
    .save("/home/jovyan/work/delta_lake/gold/gold_order_items_enriched")

print("gold_order_items_enriched:", gold_enriched.count(), "rows")
gold_enriched.show(5, truncate=False)

In [ ]:
gold_basket = (
    gold_enriched
    .groupBy("order_id")
    .agg(F.collect_set("category_english").alias("items"))  # ← đổi collect_list → collect_set
    .filter(F.size("items") >= 2)
)

gold_basket.write.format("delta").mode("overwrite") \
    .save("/home/jovyan/work/delta_lake/gold/gold_basket")

print("gold_basket:", gold_basket.count(), "đơn hàng hợp lệ")
gold_basket.show(10, truncate=False)

In [ ]:
print("=== TOP 10 CATEGORIES ===")
(gold_enriched
 .groupBy("category_english")
 .count()
 .orderBy(F.desc("count"))
 .show(10, truncate=False))

print("=== PHÂN BỐ BASKET SIZE ===")
(gold_basket
 .withColumn("basket_size", F.size("items"))
 .groupBy("basket_size")
 .count()
 .orderBy("basket_size")
 .show(15))

print("=== THỐNG KÊ ===")
(gold_basket
 .withColumn("basket_size", F.size("items"))
 .select(
     F.count("order_id").alias("total_orders"),
     F.avg("basket_size").alias("avg_basket_size"),
     F.max("basket_size").alias("max_basket_size"),
 )
 .show())

In [ ]:
# Đọc lại để verify
check = spark.read.format("delta").load(
    "/home/jovyan/work/delta_lake/gold/gold_basket"
)
print("Verify gold_basket OK — rows:", check.count())

# Xem Delta log
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, "/home/jovyan/work/delta_lake/gold/gold_basket")
dt.history(3).select("version", "timestamp", "operation").show()